In [0]:
import uuid
from pyspark.sql.functions import col

In [0]:
SILVER_PATH = "/Volumes/workspace/legal_data/silver/legal_sections/"

silver_df = spark.read.format("delta").load(SILVER_PATH)

silver_df.count()

In [0]:
def chunk_text(text, max_chars=1800, overlap=200):
    if not text:
        return []

    text = text.strip()
    chunks = []
    start = 0

    while start < len(text):
        end = start + max_chars
        chunk = text[start:end]

        # try to end at sentence boundary
        if end < len(text):
            last_period = chunk.rfind(".")
            if last_period > 300:
                end = start + last_period + 1
                chunk = text[start:end]

        chunks.append(chunk.strip())
        start = end - overlap

    return chunks

In [0]:
from collections import Counter
import re

def extract_keywords(text, top_n=5):
    words = re.findall(r'\b[a-zA-Z]{4,}\b', text.lower())
    common = Counter(words).most_common(top_n)
    return [w for w, _ in common]

In [0]:
records = []

for row in silver_df.collect():

    chunks = chunk_text(row.section_text)

    for i, chunk in enumerate(chunks):

        records.append({
            "chunk_id": str(uuid.uuid4()),
            "section_id": row.section_id,
            "doc_id": row.doc_id,
            "file_name": row.file_name,
            "category": row.category,
            "act_name": row.act_name,
            "section_number": row.section_number,
            "chunk_index": i,
            "chunk_text": chunk,
            "char_count": len(chunk),
            "keywords": extract_keywords(chunk),
            "created_at": row.created_at
        })

In [0]:
gold_df = spark.createDataFrame(records)

gold_df.display()

In [0]:
GOLD_PATH = "/Volumes/workspace/legal_data/gold/legal_chunks/"

In [0]:
gold_df.write.format("delta") \
    .mode("overwrite") \
    .save(GOLD_PATH)

gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_legal_chunks")


##### Use this after this :
gold_df.write \
    .mode("append") \
    .saveAsTable("workspace.default.gold_legal_chunks")

In [0]:
%sql
CREATE TABLE workspace.default.gold_legal_chunks (
  act_name STRING,
  category STRING,
  char_count BIGINT,
  chunk_id STRING,
  chunk_index BIGINT,
  chunk_text STRING,
  created_at TIMESTAMP,
  doc_id STRING,
  file_name STRING,
  keywords ARRAY<STRING>,
  section_id STRING,
  section_number STRING
)
USING DELTA;

In [0]:
gold_df.write.mode("append").saveAsTable("workspace.default.gold_legal_chunks")

In [0]:
%sql
select count(*) from workspace.default.gold_legal_chunks

In [0]:
gold_df.count()

In [0]:
gold_df.select("char_count").describe().display()

In [0]:
gold_df.select("chunk_text").limit(10).display()

In [0]:
gold_df.select("section_number").tail(10)